# 04 — Land Surface Predictor QC (FIXED)

Uses the adapted land set:
**DEM/Elevation, NDVI, LST Day, Distance from Sea**.

LST Night is intentionally excluded because it is not part of this Khulna implementation.
Slope/Aspect may remain in the repository but are not part of the default paper-style combinations.

This notebook does not resample training predictors.

In [1]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import re
import numpy as np
import pandas as pd
import rasterio

PRED_ROOT = RAW_DIR / "predictors"

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def rasters(folder):
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")]) if folder.exists() else []

for name in ["DEM","NDVI","LST_Day","Distance_Sea"]:
    folder = PRED_ROOT / name
    print(name, "->", len(rasters(folder)), "rasters")

DEM -> 1 rasters
NDVI -> 72 rasters
LST_Day -> 72 rasters
Distance_Sea -> 5 rasters


In [3]:
rows = []
for name in ["DEM","NDVI","LST_Day","Distance_Sea","Slope","Aspect"]:
    folder = PRED_ROOT / name
    for p in rasters(folder):
        with rasterio.open(p) as src:
            a = src.read(1, masked=True)
            v = a.compressed().astype("float64")
            rows.append({
                "predictor":name, "file":p.name, "path":str(p),
                "year_month":parse_ym(p.name),
                "crs":str(src.crs), "width":src.width, "height":src.height,
                "res_x":src.res[0], "res_y":src.res[1], "nodata":src.nodata,
                "valid_pct":100*v.size/a.size if a.size else np.nan,
                "min":float(np.nanmin(v)) if v.size else np.nan,
                "max":float(np.nanmax(v)) if v.size else np.nan,
                "mean":float(np.nanmean(v)) if v.size else np.nan,
                "bounds":str(tuple(src.bounds)),
            })

qc = pd.DataFrame(rows)
display(qc.head(30))
qc.to_csv(PROCESSED_DIR / "land_predictor_native_qc.csv", index=False)

,predictor,file,path,year_month,crs,width,height,res_x,res_y,nodata,valid_pct,min,max,mean,bounds
0,DEM,Khulna_SRTM_DEM.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,None,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",1929,5004,0.000269,0.000269,NaN,100.0,-27.0000,32.0000,2.727080,"(89.23531655788203, 21.66332223418432, 89.7551..."
1,NDVI,NDVI_2017_1.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 1)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1913,0.9295,0.453471,"(89.22965717159208, 21.658381500121664, 89.759..."
2,NDVI,NDVI_2017_10.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 10)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.2000,0.8824,0.585164,"(89.22965717159208, 21.658381500121664, 89.759..."
3,NDVI,NDVI_2017_11.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 11)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1992,0.8811,0.570727,"(89.22965717159208, 21.658381500121664, 89.759..."
4,NDVI,NDVI_2017_12.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 12)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1997,0.8979,0.522399,"(89.22965717159208, 21.658381500121664, 89.759..."
5,NDVI,NDVI_2017_2.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 2)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1895,0.9739,0.481362,"(89.22965717159208, 21.658381500121664, 89.759..."
6,NDVI,NDVI_2017_3.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 3)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1981,0.8893,0.503202,"(89.22965717159208, 21.658381500121664, 89.759..."
7,NDVI,NDVI_2017_4.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 4)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1762,0.8759,0.466229,"(89.22965717159208, 21.658381500121664, 89.759..."
8,NDVI,NDVI_2017_5.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 5)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1788,0.8715,0.434025,"(89.22965717159208, 21.658381500121664, 89.759..."
9,NDVI,NDVI_2017_6.tif,E:\Geospatial\Precipitation-Downscaling-Khulna...,"(2017, 6)","GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,0.008983,0.008983,NaN,100.0,-0.1997,0.8794,0.435640,"(89.22965717159208, 21.658381500121664, 89.759..."


In [4]:
# Dynamic monthly coverage: NDVI and LST_Day
for name in ["NDVI","LST_Day"]:
    files = rasters(PRED_ROOT/name)
    by_ym = {}
    for p in files:
        ym = parse_ym(p.name)
        if ym:
            if ym in by_ym:
                raise ValueError(f"Duplicate {name} raster for {ym}: {by_ym[ym]} and {p}")
            by_ym[ym] = p
    missing = [(y,m) for y in range(2017,2023) for m in range(1,13) if (y,m) not in by_ym]
    print(f"{name}: parsed={len(by_ym)}, missing={len(missing)}")
    if missing:
        print(missing)

# Static predictors: choose one canonical file explicitly/heuristically.
def choose_static(folder_name, preferred_names):
    folder = PRED_ROOT / folder_name
    files = rasters(folder)
    if not files:
        raise FileNotFoundError(f"No raster found for {folder_name}")
    lower = {p.name.lower():p for p in files}
    for name in preferred_names:
        if name.lower() in lower:
            return lower[name.lower()]
    # Prefer filenames that do not contain obvious intermediate terms.
    clean = [p for p in files if not any(x in p.stem.lower() for x in ["clip","tmp","temp","aligned","resampl"])]
    if len(clean) == 1:
        return clean[0]
    if len(files) == 1:
        return files[0]
    raise ValueError(
        f"Multiple ambiguous {folder_name} rasters found. Set a canonical file manually:\n" +
        "\n".join(str(p) for p in files)
    )

DEM_PATH = choose_static("DEM", ["Khulna_SRTM_DEM.tif","DEM.tif"])
DFS_PATH = choose_static("Distance_Sea", ["Distance_Sea.tif","distance_to_sea.tif"])

print("Canonical DEM:", DEM_PATH)
print("Canonical Distance-to-Sea:", DFS_PATH)

(PROCESSED_DIR / "canonical_predictors.txt").write_text(
    f"DEM={DEM_PATH}\nDistance_Sea={DFS_PATH}\n",
    encoding="utf-8"
)

NDVI: parsed=72, missing=0
LST_Day: parsed=72, missing=0
Canonical DEM: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif
Canonical Distance-to-Sea: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif


367

In [5]:
print("\n04 complete. No training raster has been resampled.")


04 complete. No training raster has been resampled.
